# ⚡ Lesson 04: XGBoost, LightGBM & CatBoost

**Time:** ~3 hours  **Prerequisites:** Lesson 03 (gradient boosting from scratch)

In Lesson 03 you built a gradient-boosting machine in 40 lines. Now meet the three libraries that took that same idea and turned it into the **default weapon of tabular machine learning**:

| library | first paper | signature idea |
|---|---|---|
| **XGBoost** | Chen & Guestrin, KDD 2016 | second-order optimisation + explicit regularisation, engineered for speed |
| **LightGBM** | Ke et al., NeurIPS 2017 | histogram splits + **leaf-wise** tree growth → very fast on large data |
| **CatBoost** | Prokhorenkova et al., NeurIPS 2018 | **ordered boosting** & ordered target statistics → principled categorical handling |

### 🎯 Learning goals
1. Explain what XGBoost adds to classic gradient boosting, and **compute its leaf weight and split gain by hand**.
2. Explain level-wise vs. leaf-wise growth, GOSS/EFB (LightGBM), and ordered boosting (CatBoost).
3. Train all three with **early stopping**, native **missing-value** and **categorical** handling.
4. Map equivalent hyperparameters across libraries.
5. Choose a library with reasons, not vibes.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

## 1 · What's different from "plain" gradient boosting?

| improvement | what it does | why it matters |
|---|---|---|
| **Second-order (Newton) step** | uses the loss's gradient **and curvature** (Hessian) | smarter, better-scaled steps; leaf values come from a closed formula |
| **Explicit regularisation** | penalises number of leaves ($\gamma$) and leaf weights ($\lambda$) *inside the objective* | less overfitting by design |
| **Shrinkage + column subsampling** | learning rate, and random feature subsets per tree/level | variance reduction, speed |
| **Histogram split finding** | bin features into ≤ 255 buckets | huge speed-ups, tiny accuracy cost |
| **Sparsity-aware splits** | each split learns a *default direction* for missing values | NaNs need no imputation |
| **Engineering** | parallel/cache-aware code, out-of-core, GPUs, early stopping | training in seconds instead of minutes |

## 2 · The XGBoost objective, step by step

XGBoost minimises

$$\mathcal{L} = \sum_{i} \ell\big(y_i, \hat y_i\big) \;+\; \sum_{k} \Omega(f_k), \qquad \Omega(f) = \gamma\,T + \tfrac12\,\lambda \sum_{j=1}^{T} w_j^2,$$

where tree $f$ has $T$ leaves with leaf weights $w_j$. *The first term is "fit the data"; the second is "keep the trees small and the leaf values modest".*

When adding tree $t$, we approximate the loss around the current prediction with a **second-order Taylor expansion**, using for each example

$$g_i = \frac{\partial \ell}{\partial \hat y_i}\;\;\text{(gradient)},\qquad h_i = \frac{\partial^2 \ell}{\partial \hat y_i^2}\;\;\text{(Hessian)}.$$

For squared error $\ell=\tfrac12(y-\hat y)^2$: $g_i=\hat y_i-y_i$ and $h_i=1$. For logistic loss: $g_i = p_i - y_i$ and $h_i = p_i(1-p_i)$.

For a fixed tree structure, let $G_j=\sum_{i\in j} g_i$ and $H_j=\sum_{i\in j} h_i$ be the sums over the examples in leaf $j$. Minimising the (quadratic) objective in $w_j$ gives:

$$\boxed{\,w_j^{*} = -\frac{G_j}{H_j + \lambda}\,}\qquad\text{and the split gain}\qquad \boxed{\,\text{Gain} = \tfrac12\Big[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\Big]-\gamma\,}$$

Read these like sentences:

* **Leaf weight:** "the average negative gradient in the leaf, but *shrunk* by $\lambda$ (a leaf with few examples (small $H$) is shrunk the most." With $\lambda=0$ and squared loss, $w^*=$ mean residual) exactly Lesson 03's leaf value.
* **Gain:** "how much better are the two children than the parent?" The $-\gamma$ means: *a split must buy at least $\gamma$ of improvement, or XGBoost won't make it* (that's pruning).

### 🧪 Worked Example 4.1: XGBoost's first tree, by hand

Reuse the five-house data from Lesson 03, but centre the target (start from the mean model, so the "current prediction" is 0 and $g_i = -(\text{residual}_i)$, $h_i = 1$):

| house | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| residual $y-\bar y$ | −5.2 | −3.2 | −4.2 | +4.8 | +7.8 |
| $g_i$ | +5.2 | +3.2 | +4.2 | −4.8 | −7.8 |
| $h_i$ | 1 | 1 | 1 | 1 | 1 |

Try the split between houses 3 and 4 with $\lambda = 1$, $\gamma = 0$:

* Left: $G_L = 12.6,\ H_L = 3$ → $w_L = -12.6/(3+1) = \mathbf{-3.15}$
* Right: $G_R = -12.6,\ H_R = 2$ → $w_R = 12.6/(2+1) = \mathbf{+4.20}$
* Gain $= \tfrac12\big[\tfrac{12.6^2}{4}+\tfrac{12.6^2}{3}-\tfrac{0^2}{6}\big] = \tfrac12[39.69+52.92] = \mathbf{46.305}$

Compare with the *unregularised* leaf means from Lesson 03 (−4.2 and +6.3): $\lambda$ has **shrunk** them by factors $3/4$ and $2/3$. Now let the machine check us, and try *every* split, not just ours:

In [2]:
x5 = np.array([[1.], [2], [3], [4], [5]])
y5 = np.array([10., 12, 11, 20, 23])
res5 = y5 - y5.mean()                       # centred target: current prediction is 0
g5, h5 = -res5, np.ones(5)                   # gradient and Hessian of squared error at prediction 0

def leaf_weight(G, H, lam):  return -G / (H + lam)
def split_gain(GL, HL, GR, HR, lam, gamma=0.0):
    score = lambda G, H: G ** 2 / (H + lam)
    return 0.5 * (score(GL, HL) + score(GR, HR) - score(GL + GR, HL + HR)) - gamma

lam = 1.0
print("candidate split after house k   ->  gain")
for k in range(1, 5):
    gain = split_gain(g5[:k].sum(), h5[:k].sum(), g5[k:].sum(), h5[k:].sum(), lam)
    print(f"   after house {k}: gain = {gain:7.3f}")

GL, HL, GR, HR = g5[:3].sum(), 3, g5[3:].sum(), 2
print(f"\nleaf weights for the best split:  left = {leaf_weight(GL, HL, lam):.3f}   right = {leaf_weight(GR, HR, lam):.3f}")

candidate split after house k   ->  gain
   after house 1: gain =   9.464
   after house 2: gain =  20.580
   after house 3: gain =  46.305
   after house 4: gain =  21.294

leaf weights for the best split:  left = -3.150   right = 4.200


In [3]:
# Now ask XGBoost itself. base_score=0 -> start from prediction 0; learning_rate=1 -> take the full Newton step;
# min_child_weight=0 -> allow tiny leaves; tree_method="exact" -> consider every threshold.
m = xgb.XGBRegressor(n_estimators=1, learning_rate=1.0, max_depth=1, reg_lambda=1.0, gamma=0.0,
                     min_child_weight=0, base_score=0.0, tree_method="exact").fit(x5, res5)

print("XGBoost predictions:", np.round(m.predict(x5), 6))
print(m.get_booster().trees_to_dataframe()[["Node", "Feature", "Split", "Gain"]].round(3).to_string(index=False))

XGBoost predictions: [-3.15 -3.15 -3.15  4.2   4.2 ]
 Node Feature  Split   Gain
    0      f0    3.5  92.61
    1    Leaf   <NA>  -3.15
    2    Leaf   <NA>    4.2


**XGBoost reproduces our hand calculation**: leaf weights $-3.15$ and $+4.20$, and a split gain of $46.305$. No magic: just the boxed formulas.

### What do $\lambda$ and $\gamma$ actually do?
* **$\lambda$ (L2 on leaf weights)** shrinks every leaf value toward 0, hardest on small leaves. Bigger $\lambda$ = more conservative model.
* **$\gamma$ (min split loss)** is a *price per split*. If the best gain is below $\gamma$ the node stays a leaf.

> ⚠️ **Read the fine print: an implementation detail worth knowing.** The *paper's* gain has a factor ½ in front. The *library* reports (and compares with `gamma`) the **un-halved** quantity: that's why the dump above says `Gain = 92.61` = 2 × 46.305. So in practice: **XGBoost splits only if its reported gain exceeds `gamma`.** Verify with the experiment below instead of trusting any blog post (including mine!).

In [4]:
print("   lambda  |  left weight  right weight")
for lam_ in [0, 1, 5, 20]:
    print(f"   {lam_:>5}   |   {leaf_weight(GL, HL, lam_):9.3f}   {leaf_weight(GR, HR, lam_):9.3f}")

print("\nXGBoost's reported gain for the best split is 92.61.  Does gamma prune it?")
for gamma_ in [0, 50, 90, 95, 200]:
    mm = xgb.XGBRegressor(n_estimators=1, learning_rate=1.0, max_depth=1, reg_lambda=1.0, gamma=gamma_, min_child_weight=0,
                          base_score=0.0, tree_method="exact").fit(x5, res5)
    n_distinct = len(np.unique(np.round(mm.predict(x5), 6)))
    print(f"  gamma = {gamma_:>3}:  {'split kept   (2 leaves)' if n_distinct == 2 else 'split PRUNED (a single leaf)'}")

   lambda  |  left weight  right weight
       0   |      -4.200       6.300
       1   |      -3.150       4.200
       5   |      -1.575       1.800
      20   |      -0.548       0.573

XGBoost's reported gain for the best split is 92.61.  Does gamma prune it?
  gamma =   0:  split kept   (2 leaves)
  gamma =  50:  split kept   (2 leaves)
  gamma =  90:  split kept   (2 leaves)
  gamma =  95:  split PRUNED (a single leaf)
  gamma = 200:  split PRUNED (a single leaf)


## 3 · XGBoost in practice

We'll use the housing data **raw**: NaNs untouched, categorical columns marked as pandas `category`. XGBoost (with `enable_categorical=True`) handles both, so there's no imputer and no one-hot encoder.

Split the training data once to get a **validation set** for early stopping. The test set stays sealed.

In [5]:
X_train, X_test, y_train, y_test = get_housing_split()

def to_category(X, reference):
    X = X.copy()
    for c in HOUSING_CATEGORICAL:
        X[c] = pd.Categorical(X[c], categories=sorted(reference[c].dropna().unique()))   # same categories everywhere
    return X

Xc_train, Xc_test = to_category(X_train, X_train), to_category(X_test, X_train)
Xt, Xv, yt, yv = train_test_split(Xc_train, y_train, test_size=0.2, random_state=SEED)   # train / validation
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

print("NaNs in the training features:", int(Xt.isna().sum().sum()), " -> we will NOT impute them")
print("dtypes:", dict(Xt.dtypes.astype(str).value_counts()))

NaNs in the training features: 397  -> we will NOT impute them
dtypes: {'int64': np.int64(8), 'float64': np.int64(6), 'category': np.int64(3)}


In [6]:
xgb_model = xgb.XGBRegressor(
    n_estimators=3000,            # generous upper bound -- early stopping will decide
    learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,          # row / column subsampling per tree
    reg_lambda=1.0, min_child_weight=1,
    tree_method="hist", enable_categorical=True,
    early_stopping_rounds=50,                      # stop when validation RMSE hasn't improved for 50 rounds
    eval_metric="rmse", random_state=SEED,
)
with Timer() as t:
    xgb_model.fit(Xt, yt, eval_set=[(Xt, yt), (Xv, yv)], verbose=False)

print(f"stopped at iteration {xgb_model.best_iteration}  (of up to 3000)   fit time {t.seconds:.1f}s")
print(f"validation RMSE: {xgb_model.best_score:.1f}      TEST RMSE: {rmse(y_test, xgb_model.predict(Xc_test)):.1f}")

hist = xgb_model.evals_result()
plt.figure(figsize=(8, 4))
plt.plot(hist["validation_0"]["rmse"], label="train"); plt.plot(hist["validation_1"]["rmse"], label="validation")
plt.axvline(xgb_model.best_iteration, color="k", ls=":", label="early-stopping point")
plt.ylim(50, 250); plt.xlabel("boosting round"); plt.ylabel("RMSE"); plt.legend(); plt.title("XGBoost learning curves"); plt.show()

stopped at iteration 192  (of up to 3000)   fit time 0.2s
validation RMSE: 113.4      TEST RMSE: 92.8


The gap between the two curves is the usual overfitting signature. Early stopping picks the round where validation error is lowest.

### The XGBoost hyperparameters you'll actually touch

| parameter | meaning | typical range |
|---|---|---|
| `n_estimators` | rounds (trees) | large (e.g. 2000) + early stopping |
| `learning_rate` (`eta`) | shrinkage per tree | 0.01–0.1 |
| `max_depth` | tree depth | 3–8 |
| `min_child_weight` | min sum of Hessian in a leaf (≈ min samples for squared loss) | 1–20 |
| `subsample` | fraction of rows per tree | 0.6–1.0 |
| `colsample_bytree` | fraction of features per tree | 0.5–1.0 |
| `reg_lambda` / `reg_alpha` | L2 / L1 on leaf weights | 0–10 / 0–1 |
| `gamma` (`min_split_loss`) | minimum gain to split | 0–5 |
| `tree_method` | `"hist"` (default, fast) / `"exact"` | hist |

## 4 · LightGBM: leaf-wise growth, GOSS and EFB

LightGBM's headline differences:

1. **Histogram splits** (like XGBoost's `hist`), but it was the pioneer.
2. **Leaf-wise (best-first) growth.** Level-wise growth splits *every* node on a level; leaf-wise growth splits **only the single leaf whose split reduces the loss the most**, wherever it is in the tree.

```
 LEVEL-WISE  (XGBoost default, sklearn)          LEAF-WISE  (LightGBM default)
          ●                                               ●
        ┌─┴─┐                                           ┌─┴─┐
        ●   ●      <- splits both nodes                 ●   ○     <- ○ = leaf, left alone
       ┌┴┐ ┌┴┐        on this level                    ┌─┴─┐
       ●  ● ●  ●                                       ●   ○      <- keeps digging where the
                                                      ┌─┴─┐          loss falls fastest
                                                      ○   ○
```

   Leaf-wise trees are lopsided and reach a given loss with **fewer leaves**: but they can overfit small datasets. So the key complexity knob is **`num_leaves`** (not depth); keep `num_leaves < 2^max_depth`.
3. **GOSS** (Gradient-based One-Side Sampling): keep all examples with large gradients (the "badly fit" ones), sample only some of the small-gradient ones, and re-weight. Faster training with little information lost.
4. **EFB** (Exclusive Feature Bundling): mutually-exclusive sparse features (never non-zero together, e.g. one-hot columns) are bundled into one: fewer features to scan.
5. **Native categorical splits**: for a category column it finds the best *partition of categories* into two groups, not just "one-vs-rest".

### LightGBM on housing

In [7]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.05,
    num_leaves=15,                 # the main complexity knob for leaf-wise trees
    min_child_samples=20,          # min rows per leaf (LightGBM's name for min_data_in_leaf)
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,   # row subsampling needs subsample_freq > 0
    reg_lambda=1.0, random_state=SEED, verbose=-1,
)
with Timer() as t:
    lgb_model.fit(Xt, yt, eval_set=[(Xv, yv)], eval_metric="rmse",
                  callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])      # categories auto-detected from dtype

print(f"stopped at iteration {lgb_model.best_iteration_}   fit time {t.seconds:.1f}s")
print(f"validation RMSE: {lgb_model.best_score_['valid_0']['rmse']:.1f}      TEST RMSE: {rmse(y_test, lgb_model.predict(Xc_test)):.1f}")

stopped at iteration 179   fit time 0.1s
validation RMSE: 114.7      TEST RMSE: 87.4


## 5 · CatBoost: ordered boosting and honest categorical encoding

CatBoost (from Yandex) targets two subtle problems:

### Problem 1: *target leakage* in categorical encoding
A popular trick for categorical features is **target encoding**: replace each category by the mean of the target in that category. Done naively on the training set, each row's own target is *part of* its feature value: the model "sees" the answer. This is **target leakage**, and it's lethal for high-cardinality categories.

CatBoost's fix: **ordered target statistics**: randomly permute the rows and encode each row using **only the rows that come *before* it** in the permutation. A row's own label never touches its own feature.

### 🧪 Worked Example 4.2: a feature that is pure noise, and a model fooled by it
We create a categorical column with 300 random levels and a target that is **completely unrelated** to it. Any model that claims to predict from it is lying.

In [8]:
rng = np.random.default_rng(1)
n_tr, n_te, n_levels, a = 1000, 1000, 300, 1.0               # a = strength of the prior
cat_tr, cat_te = rng.integers(0, n_levels, n_tr), rng.integers(0, n_levels, n_te)
y_tr, y_te = rng.normal(size=n_tr), rng.normal(size=n_te)    # target: pure noise, independent of the category!
prior = y_tr.mean()

# --- naive target encoding: category mean computed on the whole training set (includes the row's own label) ---
stats = pd.DataFrame({"c": cat_tr, "y": y_tr}).groupby("c")["y"].agg(["sum", "count"])
table = (stats["sum"] + a * prior) / (stats["count"] + a)
enc_naive_tr = table.reindex(cat_tr).values
enc_te = table.reindex(cat_te).fillna(prior).values            # test rows use the training table (no leakage there)

# --- ordered target statistics: for each row, use only the rows BEFORE it in a random permutation ---
perm = rng.permutation(n_tr)
d = pd.DataFrame({"c": cat_tr[perm], "y": y_tr[perm]})
prev_sum = d.groupby("c")["y"].cumsum() - d["y"]                # sum of earlier rows' targets in the same category
prev_cnt = d.groupby("c").cumcount()                            # number of earlier rows in the same category
enc_ord_tr = np.empty(n_tr); enc_ord_tr[perm] = ((prev_sum + a * prior) / (prev_cnt + a)).values

def r2_of(enc_tr, enc_te_):
    lin = LinearRegression().fit(enc_tr[:, None], y_tr)
    return lin.score(enc_tr[:, None], y_tr), lin.score(enc_te_[:, None], y_te)

print("                              train R²    test R²   (truth: both should be ≈ 0)")
print("naive target encoding      : %8.3f  %9.3f" % r2_of(enc_naive_tr, enc_te))
print("ordered target statistics  : %8.3f  %9.3f" % r2_of(enc_ord_tr, enc_te))

                              train R²    test R²   (truth: both should be ≈ 0)
naive target encoding      :    0.307     -0.446
ordered target statistics  :    0.001     -0.006


The naive encoding **hallucinates a relationship** on the training set (R² ≈ 0.3) that not only evaporates on test data but turns *negative* (the model is confidently wrong): a classic leak. The ordered version honestly reports "no signal" on both. CatBoost builds this protection into the algorithm.

### Problem 2: *prediction shift* in boosting itself
In ordinary boosting, each tree's residuals are computed using a model that was trained on **the same rows**, so residuals are slightly too optimistic: a subtle bias CatBoost calls *prediction shift*. **Ordered boosting** maintains models trained on only the earlier part of a random permutation, so each example's residual comes from a model that has never seen it.

### Other CatBoost traits
* **Symmetric ("oblivious") trees:** every node at the same depth uses the *same* split. That regularises strongly and makes prediction extremely fast.
* **Excellent defaults**: often competitive with *no tuning*.
* Pass `cat_features=[...]` (strings or ints) and it does the rest. Missing numeric values are handled natively; for categorical NaNs we'll use a `"missing"` label.

In [9]:
# CatBoost likes categorical columns as plain strings
Xs_train, Xs_test = X_train.copy(), X_test.copy()
for c in HOUSING_CATEGORICAL:
    Xs_train[c] = Xs_train[c].fillna("missing").astype(str)
    Xs_test[c] = Xs_test[c].fillna("missing").astype(str)
Xst, Xsv, yst, ysv = train_test_split(Xs_train, y_train, test_size=0.2, random_state=SEED)   # same split as before

cat_model = CatBoostRegressor(
    allow_writing_files=False,
    iterations=3000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
    cat_features=HOUSING_CATEGORICAL, early_stopping_rounds=50,
    random_seed=SEED, verbose=0,
)
with Timer() as t:
    cat_model.fit(Xst, yst, eval_set=(Xsv, ysv))

print(f"stopped at iteration {cat_model.get_best_iteration()}   fit time {t.seconds:.1f}s")
print(f"TEST RMSE: {rmse(y_test, cat_model.predict(Xs_test)):.1f}")

stopped at iteration 563   fit time 1.3s
TEST RMSE: 88.2


## 6 · The head-to-head

All five boosters, same split, same early-stopping budget (50 rounds), same learning rate where applicable. Remember: the **test set was never used to make any decision**.

In [10]:
def hgb_fit():
    return HistGradientBoostingRegressor(learning_rate=0.05, max_iter=3000, max_depth=4, categorical_features="from_dtype",
                                         early_stopping=True, validation_fraction=0.2, n_iter_no_change=50, random_state=SEED).fit(Xc_train, y_train)

with Timer() as t_x:
    xm = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                          enable_categorical=True, early_stopping_rounds=50, random_state=SEED).fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
with Timer() as t_l:
    lm = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=15, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=SEED, verbose=-1).fit(Xt, yt, eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(50, verbose=False)])
with Timer() as t_c:
    cm = CatBoostRegressor(allow_writing_files=False, iterations=3000, learning_rate=0.05, depth=6, cat_features=HOUSING_CATEGORICAL, early_stopping_rounds=50,
                           random_seed=SEED, verbose=0).fit(Xst, yst, eval_set=(Xsv, ysv))
with Timer() as t_h:
    hm = hgb_fit()

bench = pd.DataFrame([
    {"model": "XGBoost",  "rounds": xm.best_iteration + 1, "fit_s": t_x.seconds, **regression_report(y_test, xm.predict(Xc_test))},
    {"model": "LightGBM", "rounds": lm.best_iteration_,    "fit_s": t_l.seconds, **regression_report(y_test, lm.predict(Xc_test))},
    {"model": "CatBoost", "rounds": cm.get_best_iteration() + 1, "fit_s": t_c.seconds, **regression_report(y_test, cm.predict(Xs_test))},
    {"model": "sklearn HistGB", "rounds": hm.n_iter_,      "fit_s": t_h.seconds, **regression_report(y_test, hm.predict(Xc_test))},
]).sort_values("RMSE").reset_index(drop=True)
bench.round(2)

,model,rounds,fit_s,RMSE,MAE,R2,MAPE_%
0,CatBoost,564,1.40,88.20,52.18,0.94,11.97
1,LightGBM,181,0.08,88.87,54.87,0.94,12.86
2,XGBoost,193,0.15,92.79,55.13,0.93,12.54
3,sklearn HistGB,280,0.27,95.21,57.15,0.93,12.95


Remember Lesson 01's lesson about noise floors: on 3,000 rows with an irreducible noise level, the **differences between modern boosters are small** compared with the difference between *boosting and everything else*. Don't over-read a few RMSE points from one split: that's exactly the trap Lesson 06 teaches you to avoid with cross-validation.

### Where the speed differences show up: bigger data
On 3,000 rows everything is instant. Let's time the libraries on 60,000 rows × 30 features with a fixed 200 rounds (no early stopping, so the work is identical).

In [11]:
from sklearn.datasets import make_regression
Xbig, ybig = make_regression(n_samples=60_000, n_features=30, n_informative=20, noise=10, random_state=0)

timings = {}
for name, make in {
    "XGBoost (hist)": lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist"),
    "LightGBM":       lambda: lgb.LGBMRegressor(n_estimators=200, num_leaves=63, learning_rate=0.1, verbose=-1),
    "CatBoost":       lambda: CatBoostRegressor(allow_writing_files=False, iterations=200, depth=6, learning_rate=0.1, verbose=0),
    "sklearn HistGB": lambda: HistGradientBoostingRegressor(max_iter=200, max_leaf_nodes=63, learning_rate=0.1, early_stopping=False),
}.items():
    with Timer() as t:
        make().fit(Xbig, ybig)
    timings[name] = t.seconds

plt.figure(figsize=(7.5, 3.4))
plt.barh(list(timings), list(timings.values()), color=[PALETTE["blue"], PALETTE["green"], PALETTE["orange"], PALETTE["gray"]])
for i, v in enumerate(timings.values()): plt.text(v + 0.1, i, f"{v:.1f}s", va="center")
plt.gca().invert_yaxis(); plt.xlabel("seconds to train 200 rounds (60k × 30)"); plt.title("Training time on a bigger problem"); plt.show()

(Your exact numbers depend on your CPU and core count; the *ordering* and the order of magnitude are what matter. On GPUs all three can be much faster still.)

## 7 · Rosetta stone: the same knob in three languages

| concept | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| number of rounds | `n_estimators` | `n_estimators` | `iterations` |
| learning rate | `learning_rate` (`eta`) | `learning_rate` | `learning_rate` |
| tree complexity | `max_depth` | **`num_leaves`** (+ `max_depth`) | `depth` (symmetric) |
| min data in a leaf | `min_child_weight` (Hessian sum) | `min_child_samples` | `min_data_in_leaf` |
| row subsampling | `subsample` | `subsample` + `subsample_freq` | `subsample` (with `bootstrap_type`) |
| feature subsampling | `colsample_bytree` | `colsample_bytree` (`feature_fraction`) | `rsm` |
| L2 on leaves | `reg_lambda` | `reg_lambda` | `l2_leaf_reg` |
| min gain to split | `gamma` | `min_split_gain` | - |
| categorical features | `enable_categorical=True` + `category` dtype | `category` dtype / `categorical_feature` | `cat_features=[...]` |
| early stopping | `early_stopping_rounds` (constructor) | `lgb.early_stopping(...)` callback | `early_stopping_rounds` |
| missing values | native (learned default direction) | native | native |

## 8 · Which should you pick?

| situation | reach for |
|---|---|
| General tabular problem, want a robust, well-documented default | **XGBoost** or **LightGBM** |
| Large data (≥ 100k rows), many features, iteration speed matters | **LightGBM** |
| Many **categorical** features, high cardinality, little time to tune | **CatBoost** |
| Small data, want strong results with *minimal* tuning | **CatBoost** |
| You need to ship with only scikit-learn installed | `HistGradientBoosting*` |
| Maximum accuracy | **Try all three**, tune each (Lesson 07), and compare with cross-validation (Lesson 06); stack them (Lesson 05) |

> 💡 Honest summary: with equal tuning effort the three libraries land within a hair of each other on most datasets. Choose by **data size, categorical content and ecosystem**, then spend your effort on *features, validation and tuning*: those matter far more than the library.

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. With squared loss and λ = 0, what is XGBoost's optimal leaf weight, and how does it relate to Lesson 03?**

<details><summary>Show answer</summary>

$w^*=-G/H$ with $g_i=\hat y_i-y_i$ and $h_i=1$, so $w^*=\frac{\sum(y_i-\hat y_i)}{n_{\text{leaf}}}$ = the **mean residual in the leaf**: exactly the leaf value of Lesson 03's gradient boosting. XGBoost generalises it to other losses and adds λ-shrinkage.

</details>

**Q2. A split has (library-reported) gain 3.2 but γ = 5. What happens?**

<details><summary>Show answer</summary>

The gain doesn't exceed γ, so XGBoost **does not make the split**: the node stays a leaf. γ acts as a per-split price, pruning splits that don't improve the objective enough. (And remember the implementation detail from Section 2: compare γ with the *reported* gain, which is 2× the ½-formula.)

</details>

**Q3. Why does LightGBM use `num_leaves` instead of (just) `max_depth`?**

<details><summary>Show answer</summary>

Leaf-wise trees are lopsided: the same depth can hold anywhere from a few leaves to $2^{depth}$. Complexity is better controlled by the *number of leaves*. A rule of thumb: `num_leaves < 2**max_depth`.

</details>

**Q4. In one sentence, why is naive target encoding dangerous and what does CatBoost do instead?**

<details><summary>Show answer</summary>

Each training row's own label leaks into its encoded feature, so the model overfits to a signal that won't exist at prediction time; CatBoost uses **ordered target statistics**, encoding each row only from rows that precede it in a random permutation.

</details>

**Q5. You see validation RMSE bottom out at round 340 and then rise slowly. Should you worry that early stopping chose round 340 using the validation set, then report the validation RMSE as your final result?**

<details><summary>Show answer</summary>

Yes, the validation score is **optimistically biased** because it was used to choose the stopping round. Report the score on a *separate* test set (or via nested cross-validation, Lesson 06).

</details>

## 9 · Problems

Shared variables: `x5, y5, res5, g5, h5`, `leaf_weight`, `split_gain`, `Xt, Xv, yt, yv, Xc_test, y_test`, `Xst, Xsv, yst, ysv, Xs_test`, `rmse`.

### 🧩 Problem 04.1 ⭐⭐: XGBoost arithmetic with a bigger λ

Using the five-house gradients `g5`, `h5` and the split after house 3, compute with **λ = 2** and **γ = 0**:

* `w_left`, `w_right`: the optimal leaf weights
* `gain`: the split gain

Use the helper functions `leaf_weight(G, H, lam)` and `split_gain(GL, HL, GR, HR, lam, gamma)`: or write the formulas yourself!

<details><summary>💡 Hint 1</summary>

Left leaf = houses 1–3: `GL = g5[:3].sum()`, `HL = h5[:3].sum()`. Right leaf = houses 4–5.

</details>

<details><summary>💡 Hint 2</summary>

`w = -G / (H + lam)`.

</details>

<details><summary>💡 Hint 3</summary>

Gain = `0.5 * (GL**2/(HL+lam) + GR**2/(HR+lam) - (GL+GR)**2/(HL+HR+lam)) - gamma`. Since GL + GR = 0 here, the last fraction vanishes.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
lam2 = 2.0
w_left = TODO    # YOUR CODE HERE
w_right = TODO   # YOUR CODE HERE
gain = TODO      # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("04.1", lambda: abs(w_left + 2.52) < 1e-6 and abs(w_right - 3.15) < 1e-6 and abs(gain - 35.721) < 1e-3,
      "Expected w_left = -12.6/5 = -2.52, w_right = 12.6/4 = 3.15, gain ≈ 35.721.", needs=[w_left, w_right, gain])

### 🧩 Problem 04.2 ⭐⭐: XGBoost with early stopping

Train an `XGBRegressor` with: `n_estimators=3000, learning_rate=0.05, max_depth=3, tree_method="hist", enable_categorical=True, early_stopping_rounds=50, random_state=SEED`.
Fit on `(Xt, yt)` with `eval_set=[(Xv, yv)]`.

Store the chosen number of rounds (`.best_iteration`) in `xgb_best_iter` and the **test RMSE** in `xgb_test_rmse`.

<details><summary>💡 Hint 1</summary>

Follow Section 3. Early stopping needs both `early_stopping_rounds` in the constructor and an `eval_set` in `.fit`.

</details>

<details><summary>💡 Hint 2</summary>

`model.best_iteration` is available after fitting with early stopping. Predict with `model.predict(Xc_test)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
xgb_best_iter = TODO    # YOUR CODE HERE
xgb_test_rmse = TODO    # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("04.2", lambda: 20 < xgb_best_iter < 2999 and xgb_test_rmse < 115,
      "Early stopping should halt well before 3000 rounds, and the test RMSE should be below 115.", needs=[xgb_best_iter, xgb_test_rmse])

### 🧩 Problem 04.3 ⭐⭐: Tune LightGBM's num_leaves

For `num_leaves` in `[4, 8, 15, 31, 63, 127]`, train an `LGBMRegressor(n_estimators=3000, learning_rate=0.05, random_state=SEED, verbose=-1)`
with early stopping (50 rounds) on `(Xv, yv)`.

* store each setting's **best validation RMSE** in the dict `val_by_leaves`,
* store the winner in `best_leaves`,
* store the winner's **test RMSE** in `lgb_test_rmse`.

*Reflection:* does a bigger `num_leaves` always help? Why might it hurt on 2,400 training rows?

<details><summary>💡 Hint 1</summary>

Use `lgb.early_stopping(50, verbose=False)` in `callbacks`. The validation RMSE is `m.best_score_['valid_0']['rmse']`.

</details>

<details><summary>💡 Hint 2</summary>

Keep the fitted models in a dict so you can predict on the test set with the winner without refitting.

</details>

<details><summary>💡 Hint 3</summary>

`best_leaves = min(val_by_leaves, key=val_by_leaves.get)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
val_by_leaves = {}
for nl in [4, 8, 15, 31, 63, 127]:
    pass   # YOUR CODE HERE

best_leaves = TODO      # YOUR CODE HERE
lgb_test_rmse = TODO    # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("04.3", lambda: len(val_by_leaves) == 6 and best_leaves == min(val_by_leaves, key=val_by_leaves.get) and lgb_test_rmse < 120,
      "Fill val_by_leaves with 6 entries; best_leaves must be its argmin; the test RMSE should be below 120.",
      needs=[best_leaves, lgb_test_rmse])

### 🧩 Problem 04.4 ⭐⭐⭐: Does native categorical handling help CatBoost?

Train **two** `CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)` models on the *full* training set:

* **native**: raw string features `Xs_train` with `cat_features=HOUSING_CATEGORICAL`
* **one-hot**: the numeric matrix from `make_preprocessor(X_train)` (impute + one-hot): no `cat_features`

Store the two **test RMSEs** in `rmse_native` and `rmse_onehot`. Then write two sentences: which won, and why might that be?

<details><summary>💡 Hint 1</summary>

Native: `CatBoostRegressor(..., cat_features=HOUSING_CATEGORICAL).fit(Xs_train, y_train)`; predict on `Xs_test`.

</details>

<details><summary>💡 Hint 2</summary>

One-hot: `prep = make_preprocessor(X_train)`; `A = prep.fit_transform(X_train)`, `B = prep.transform(X_test)`, then fit on `A` and predict on `B`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
rmse_native = TODO    # YOUR CODE HERE
rmse_onehot = TODO    # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("04.4", lambda: 60 < rmse_native < 130 and 60 < rmse_onehot < 130,
      "Both models should land between roughly 60 and 130 RMSE.", needs=[rmse_native, rmse_onehot])

In [16]:
scoreboard()

Scoreboard: 4 solved, 0 to fix, 0 not attempted
   ✅ 04.1
   ✅ 04.2
   ✅ 04.3
   ✅ 04.4


## 🏁 Recap

* XGBoost = gradient boosting + **second-order** optimisation + **explicit regularisation**: $w^*=-G/(H+\lambda)$, Gain $=\tfrac12[\dots]-\gamma$. We verified both by hand.
* **LightGBM**: histograms, **leaf-wise** growth (`num_leaves`), GOSS, EFB: superb on large data.
* **CatBoost**: **ordered** boosting and ordered target statistics → leak-free categoricals; symmetric trees; strong defaults.
* All three handle **missing values natively**, support **early stopping**, and land close together in accuracy when tuned.
* Choose by data size, categorical content and ecosystem; invest effort in validation, features and tuning.

### ➡️ Next up: **Lesson 05: Voting & Stacking**
What if, instead of choosing one library, we combine **all** of them (along with models of completely different kinds) and *learn* how to weight them?